# Data Preprocessing and Feature Engineering (Part II)

## Data Processing Steps
### Step 4: Detect & Handle Outliers
---
```python

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Get numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Figure for outlier detection
plt.figure(figsize=(15, 10))
sns.boxplot(data=df_encoded[numeric_cols])
plt.title('Boxplot of Numeric Features')
plt.xticks(rotation=45)
plt.show()

# Function to detect outliers using IQR
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[column] < lower_bound) | (data[column] > upper_bound)]        # Outliers based on IQR

# Detect outliers for each numeric column
outliers = {}
for col in numeric_cols:
    outliers[col] = detect_outliers_iqr(df_encoded, col)    

# Print outliers
for col, outlier_data in outliers.items():
    if not outlier_data.empty:
        print(f"Outliers in {col}:")
        print(outlier_data)
```
Upon detecting outliers, now choose how to handle them:

Options include:
- Remove outliers: In this case, the rows containing outliers are removed from the dataset. Use this method when outliers are likely to be errors or irrelevant to the analysis. Coution: This may lead to loss of valuable data.
- Cap or floor outliers: Use this method to limit the extreme values to a certain percentile (e.g., 1st and 99th percentiles). The Outliers are replaced by the closest capped value. This is useful when all data points are important, but extreme values need to be controlled.
- Transform data: Apply transformations like log, square root, or Box-Cox to reduce the impact of outliers. This is useful when the data is skewed.

```python
# Removing outliers
def remove_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[column] = data[(data[column] >= lower_bound) & (data[column] <= upper_bound)]
    return data

# Remove outliers for each numeric column
for col in numeric_cols:
    df_no_outliers = remove_outliers(df_encoded, col)

# Capping outliers
def cap_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[column] = data_encoded[column].clip(lower=lower_bound, upper=upper_bound)
    return data

# Cap outliers for each numeric column
for col in numeric_cols:
    df_capped = cap_outliers(df_encoded, col)

# Transforming data (Log Transformation)
def log_transform(data, column):
    data[column] = np.log1p(data[column])  # log1p is used to handle log(0)
    return data 

# Apply log transformation for each numeric column
for col in numeric_cols:
    df_transformed = log_transform(df_encoded, col)
```
### Step 5: Feature Transformation
---
Feature transformation means modifying existing features to improve model performance.

Method for Feature Transformation:
- Polynomial Features: Create new features by raising existing features to a power or multiplying them together. This is useful for capturing non-linear relationships & interactions between features.
- Binning: Convert continuous features into categorical features by dividing them into bins or intervals.
- Domain-Specific Transformations: Apply transformations based on domain knowledge, such as converting temperature from Celsius to Fahrenheit.

```python
# Polynomial Features
from sklearn.preprocessing import PolynomialFeatures

poly_cols = ['feature1', 'feature2']  # Specify columns for polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)         # include_bias=False to avoid adding a column of ones.  Why? Because it can lead to multicollinearity(highly correlated features) issues in regression models. degree=2 means we are creating squared and interaction terms.

poly_features = poly.fit_transform(df_encoded[poly_cols])
poly_feature_names = poly.get_feature_names_out(poly_cols)
# This will create new feature names like 'feature1', 'feature2', 'feature1^2', 'feature1 feature2', 'feature2^2'

df_poly = pd.DataFrame(poly_features, columns=poly_feature_names)
df_encoded = pd.concat([df_encoded, df_poly], axis=1)

# Binning: Convert continuous feature into categorical feature. Like a colume of Age can be categorized into bins like 0-10 as child, 10-20 as teenager, etc.
df_encoded['binned_feature'] = pd.cut(df_encoded['continuous_feature'], bins=[0, 10, 20, 30, 40], labels=['label1', 'label2', 'label3', 'label4'])          # This will create a new categorical feature based on the specified bins.

# Domain-Specific Transformation: Convert numerical feature into categorical feature based on the feature domain knowledge.
df_encoded['transformed_feature'] = df_encoded['numerical_feature'].apply(lambda x: 'high' if x > threshold else 'low')  # Replace 'threshold' with an appropriate value based on domain knowledge.
```
### Step 6: Preprocessing Pipeline
---
A pipeline in ML ensures preprocessing and modeling steps run in a fixed sequence without mixing training and test data.
The Pipeline flow: Data -> Missing Value Handling -> Encoding -> Scaling/Normalization -> Model Training

```python
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression

# Numerical pipeline: In this pipeline, we scale numerical features using StandardScaler.
num_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())                           # Scaling
])      # Here no numeric cols is specified, it will be specified in ColumnTransformer. Here only scaling step is defined.

# Categorical pipeline: In this pipeline, we apply One-Hot Encoding to categorical features.
cat_pipeline = Pipeline(steps=[
    ('ohe', OneHotEncoder(handle_unknown='ignore'))     # One-Hot Encoding
])
# As same as numeric pipeline, no categorical columns specified here.

# Combine pipelines: Combine numerical and categorical pipelines using ColumnTransformer.
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, numeric_cols),
    ('cat', cat_pipeline, categorical_columns)
])
# Here numerical and categorical columns are specified & transforemed using their respective pipelines.

# Final pipeline with model
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])      # Preprocessor handles all preprocessing steps before model training, ensuring no data leakage. Classifier is the ML model to be trained.
```
### Step 7: Train-Test Split & Model Evaluation
---

```python
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Split data into features and target variable
X = df_encoded.drop('target_column', axis=1)
y = df_encoded['target_column']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Train the model
clf.fit(X_train, y_train)

# Predict & evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
```